In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Project paths
source_dir = "/Workspace/Users/niranjanebi706@gmail.com/telecom_project/sample_data"

# Generate a unique batch ID for this ingestion run
batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

print("Bronze ingestion started")
print(f"Source directory: {source_dir}")
print(f"Batch ID: {batch_id}")

In [0]:
# ============================================================
# SOURCE TABLE CONFIGURATION
# ============================================================

source_tables = [
    "customers",
    "customer_addresses",
    "customer_contacts",
    "mobile_plans",
    "service_types",
    "plan_services",
    "subscriptions",
    "subscription_services",
    "call_records",
    "sms_records",
    "data_usage",
    "bills",
    "bill_items",
    "payments",
    "complaint_categories",
    "complaints",
    "service_areas"
]

print(f"Total source tables: {len(source_tables)}")
print("\nSource tables:")

for i, table in enumerate(source_tables, start=1):
    print(f"{i:02d}. {table}")

In [0]:
# ============================================================
# TEST SOURCE FILE
# ============================================================

test_table = "customers"
test_path = f"{source_dir}/{test_table}.csv"

test_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(test_path)
)

print(f"Table: {test_table}")
print(f"Path: {test_path}")
print(f"Rows: {test_df.count():,}")

print("\nColumns:")
print(test_df.columns)

print("\nSample records:")
display(test_df.limit(5))

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS telecom;

CREATE SCHEMA IF NOT EXISTS telecom.bronze;
CREATE SCHEMA IF NOT EXISTS telecom.silver;
CREATE SCHEMA IF NOT EXISTS telecom.gold;
CREATE SCHEMA IF NOT EXISTS telecom.quality;
CREATE SCHEMA IF NOT EXISTS telecom.control;

SHOW SCHEMAS IN telecom;

In [0]:
# ============================================================
# BRONZE INGESTION — ALL 17 TABLES
# ============================================================

bronze_counts = {}

for table_name in source_tables:

    print(f"\nProcessing: {table_name}")

    file_path = f"{source_dir}/{table_name}.csv"

    # Read raw CSV
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
    )

    # Add Bronze ingestion metadata
    bronze_df = (
        df
        .withColumn("_source_table", F.lit(table_name))
        .withColumn("_source_file", F.lit(f"{table_name}.csv"))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    # Write to Bronze as Delta
    target_table = f"telecom.bronze.{table_name}"

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    row_count = bronze_df.count()
    bronze_counts[table_name] = row_count

    print(f"✅ {target_table} → {row_count:,} rows")

print("\n" + "=" * 80)
print("BRONZE INGESTION COMPLETED")
print("=" * 80)